# Misinformation Models  — Opinion Classification
Driver notebook: runs each model, collects predictions, compares metrics.  
Add new models in the **Run models** cell. Train/dev only — test set not touched.
Had help from Claude on implementation.

### Import configuration and packages ###

In [13]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import torch
import cnn_baseline as cnn

#Running this will import FastText vector file, which is stored on HuggingFace and is >4gb. 
from config import DATA_DIR, FASTTEXT_PATH, TARGETS

from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis

DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"Device: {DEVICE}")
print(f"Targets: {TARGETS}")

Device: mps
Targets: ['opinion_label', 'misinformation_label']


In [14]:
# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


### Run models ###


In [15]:
# Each model should produce (preds, labels, probs) and store them in `results`.
# Add new models below following the same pattern.
results = {}

In [16]:
# Run CNN model here
# TextCNN — trains on both targets simultaneously, then evaluates each separately
vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)
cnn_model    = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model    = cnn.train_model(cnn_model, train_loader, dev_loader, train_rows, DEVICE)

for target in TARGETS:
    results[f"TextCNN — {target}"] = cnn.predict(cnn_model, dev_loader, DEVICE, target=target)

Loading FastText vectors from /Users/jennifer/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3875/4091 vocab tokens found in FastText vectors (94.7%)
Epoch   1 | loss=1.7707 | dev_macro_f1=0.6762 (primary: opinion_label)
Epoch   2 | loss=1.5694 | dev_macro_f1=0.6726 (primary: opinion_label)
Epoch   3 | loss=1.3942 | dev_macro_f1=0.6784 (primary: opinion_label)
Epoch   4 | loss=1.2303 | dev_macro_f1=0.6788 (primary: opinion_label)
Epoch   5 | loss=1.0714 | dev_macro_f1=0.6800 (primary: opinion_label)
Epoch   6 | loss=0.9486 | dev_macro_f1=0.6875 (primary: opinion_label)
Epoch   7 | loss=0.8385 | dev_macro_f1=0.7076 (primary: opinion_label)
Epoch   8 | loss=0.7206 | dev_macro_f1=0.6937 (primary: opinion_label)
Epoch   9 | loss=0.6353 | dev_macro_f1=0.6808 (primary: opinion_label)
Epoch  10 | loss=0.5526 | dev_macro_f1=0.6808 (primary: opinion_label)
Epoch  11 | loss=0.5017 | dev_macro_f1=0.7017 (primary: o

In [17]:
# Run Logistic Regression model — trained separately per target
import logreg_baseline as lr

for target in TARGETS:
    results[f"LogReg — {target}"] = lr.run(train_rows, dev_rows, task=target)


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7029

── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8590


In [18]:
#Create table to compare models across metrics
rows = []
for name, (preds, labels, probs) in results.items():
    m = compute_metrics(preds, labels, probs)
    rows.append({"Model": name, "Accuracy": m["accuracy"], "Macro F1": m["macro_f1"],
                 "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
                 "AUC-ROC": m.get("auc_roc", float("nan"))})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(rows).set_index("Model")

,Accuracy,Macro F1,F1 (not-op),F1 (opinion),AUC-ROC
Model,,,,,
TextCNN — opinion_label,0.7100,0.7076,0.7339,0.6813,0.7669
TextCNN — misinformation_label,0.8800,0.8331,0.9216,0.7447,0.9406
LogReg — opinion_label,0.6900,0.6829,0.7304,0.6353,0.7527
LogReg — misinformation_label,0.9050,0.8707,0.9373,0.8041,0.9567


In [19]:
#Model details
for name, (preds, labels, probs) in results.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print_confusion_matrix(preds, labels)
    print()
    print_sklearn_report(preds, labels)
    print()
    error_analysis(dev_rows, preds, labels)


TextCNN — opinion_label
Confusion matrix (rows=true, cols=predicted):
                 pred=0  pred=1
  true=0 (not-op):    80      37
  true=1 (opinion):   21      62

              precision    recall  f1-score   support

 not-opinion       0.79      0.68      0.73       117
     opinion       0.63      0.75      0.68        83

    accuracy                           0.71       200
   macro avg       0.71      0.72      0.71       200
weighted avg       0.72      0.71      0.71       200


False Positives (predicted opinion, actually not) — 5 shown:
  [17] 'Birds in a blizzard. We put out extra sunflower seeds since their usual food sources just got‚Ä¶ https://www.instagram.c'
  [25] "I'm getting used to seeing a layer of ash on my car each morning - and I'm in AB. We've  only seen a red sun this week. "
  [55] "@jihettly @esd2000 good morning! Yes I'm in the middle of the blizzard. I have plenty of food so I'm happy! Lol. Stay wa"
  [59] 'Re: Hurricane Matthew: All of the @ASUTenni